In [ ]:
# %pip install requests  # uncomment this line if requests is not yet installed
import json
import requests
from typing import List, Dict

print("Packages imported successfully!")


: 

## 2️⃣ Verify Ollama Server is Running

A quick `GET` request to `/api/tags` will return a list of available models if the server is up.  
If you see an error here, start your local Ollama instance first.


In [ ]:
OLLAMA_URL = "http://localhost:11434"

def check_server() -> None:
    try:
        resp = requests.get(f"{OLLAMA_URL}/api/tags")
        resp.raise_for_status()
        print("Ollama is running! Available models:")
        for model in resp.json()["models"]:
            print(f" - {model['name']}")
    except Exception as e:
        print("Could not reach Ollama server:", e)

check_server()


: 

## 3️⃣ Helper Function – `chat_with_ollama()`

This function takes a list of messages (`messages`) and the model name, sends them to the `/api/chat` endpoint, and returns the assistant’s reply.  
We also return the updated conversation context so students can keep adding more turns.

**Message format**

```json
[
  {"role":"user","content":"..."},
  {"role":"assistant","content":"..."},
  ...
]


In [ ]:

def chat_with_ollama(
    messages: List[Dict[str, str]],
    model: str = "llama3"
) -> Dict[str, str]:
    """
    Sends the conversation `messages` to Ollama and returns a dict:
        {
            'role': 'assistant',
            'content': '<reply>'
        }
    Also raises an exception on HTTP or JSON errors.
    """
    payload = {
        "model": model,
        "messages": messages
    }

    try:
        resp = requests.post(f"{OLLAMA_URL}/api/chat", json=payload)
        resp.raise_for_status()
    except requests.HTTPError as http_err:
        raise RuntimeError(f"HTTP error: {http_err}") from None
    except Exception as err:
        raise RuntimeError(f"Request failed: {err}") from None

    result = resp.json()

    # Ollama returns {"message": {...}}; we just pull out the content.
    reply = result["message"]
    return reply


## 4️⃣ Start a Fresh Conversation

We’ll begin by asking a simple question.  
After receiving the answer, we append both user and assistant messages to our conversation context so that subsequent turns see the full history.

> 👩‍🏫 **Tip:** Think of `messages` as the chat log you are sending back‑and‑forth.


In [ ]:
# Start an empty context
conversation: List[Dict[str, str]] = []

# 1st user message
user_msg_1 = "Hi! Can you explain what a neural network is?"
conversation.append({"role": "user", "content": user_msg_1})

# Get assistant reply
assistant_reply_1 = chat_with_ollama(conversation)

# Append assistant's response to context
conversation.append(assistant_reply_1)

print("Assistant:", assistant_reply_1["content"])


## 5️⃣ Add a Follow‑Up Turn

Now that we have the initial answer, ask for clarification.  
Notice how the assistant’s reply includes earlier context for reference.




In [ ]:
# 2nd user message (clarification)
user_msg_2 = "What does it mean when you say 'layers' in this context?"
conversation.append({"role": "user", "content": user_msg_2})

assistant_reply_2 = chat_with_ollama(conversation)
conversation.append(assistant_reply_2)

print(f"Conversation Length: {len(conversation)} messages")

print("Assistant:", assistant_reply_2["content"])


## 6️⃣ Build a Longer Thread

Let’s keep the conversation going for a few more turns.  
You can paste your own questions below or let me know what you'd like to ask!

> 🚀 **Experiment:** Try changing `model` in `chat_with_ollama()` to `"mistral"` or any other model you have installed.


In [ ]:
# Optional: add more turns manually
additional_questions = [
    "How does back‑propagation work?",
    "Can you give me a simple example in Python?"
]

for q in additional_questions:
    conversation.append({"role": "user", "content": q})
    reply = chat_with_ollama(conversation)
    conversation.append(reply)
    print("\nUser:", q)
    print("Assistant:", reply["content"])


## 8️⃣ Bonus – Streaming Responses (Optional)

If you want to see the answer as it is generated, you can enable streaming.  
This requires a slightly different request and handling of the response stream.

> ⚠️ This cell is optional; uncomment only if you are comfortable with low‑level HTTP streams.


In [ ]:
# Uncomment to try streaming (requires an updated Ollama server that supports it)
"""
def chat_stream(messages: List[Dict[str, str]], model: str = "llama3"):
    payload = {"model": model, "messages": messages}
    stream_resp = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json=payload,
        stream=True
    )
    for chunk in stream_resp.iter_lines():
        if chunk:
            part = json.loads(chunk)
            # Ollama streams lines like {"message":{"role":"assistant","content":"..."}}
            yield part["message"]["content"]

# Example usage
conversation_stream: List[Dict[str, str]] = [
    {"role": "user", "content": "Tell me a joke about cats."}
]
for token in chat_stream(conversation_stream):
    print(token, end='', flush=True)
"""
